# **Databases**

### Debeaver Database Setup

Create a database connection

Click:

Database → New Database Connection

You'll get a big list of database types.

Since you're familiar with SQL Server 2022 at work, I'd actually suggest choosing:

SQL Server

That way we can build you a little SQL Server practice environment and your experience will translate pretty directly between work and home.

DBeaver will ask for things like:

Host:       ?
Port:       1433
Database:   ?
Username:   ?
Password:   ?

What I'd do in your lab

This gives us another nice little project. We can run SQL Server in a container/VM in your homelab, then use DBeaver on your CachyOS laptop as the management interface.

The architecture would simply be:

DBeaver on your laptop → network → SQL Server in your lab

Then you could practice the same kinds of things you do at work:
CREATE DATABASE LabDB;

CREATE TABLE Servers (
    id INT PRIMARY KEY,
    hostname VARCHAR(50),
    ip_address VARCHAR(15),
    operating_system VARCHAR(50)
);

INSERT INTO Servers
VALUES (1, 'k8s-controller', '10.0.0.248', 'Oracle Linux');
SELECT * FROM Servers;

And DBeaver will give you the graphical table/database explorer and query-results grid you're accustomed to.

We could even make the database about your homelab infrastructure, which would make learning SQL a little more interesting than creating random customers and employees tables.

For now, open DBeaver and get to the New Database Connection screen. We can build the SQL Server side next and connect the two.

### Redis

#### Definitions
SET command: is used to set the string value of a key
    For example, let's set a key named mykey with the value Hello Redis

#### Installation
sudo apt update -y
sudo apt install -y redis-server
which redis-server
sudo service redis-server start

#### CLI
* redis-cli 
    ping << to test connectivity should return pong
    SET mykey "Hello Redis"
    quit

* redis-cli
    GET mykey
    quit

* redis-cli
    ping
    SET mykey myvalue
    GET mykey << should return "myvalue" >>
    SET user:1001 John
    GET user:1001
    exit

* redis-cli
    EXISTS mykey >> Redis will respond with: (integer) 1 // A response of 1 indicates that the key mykey exists in Redis.
    EXISTS nonexistentkey >> Redis will respond with: (integer) 0 // A response of 0 indicates that the key nonexistentkey does not exist in Redis
    exit

* redis-cli
    DEL mykey >> A response of (integer) 1 indicates that one key was successfully deleted
    EXISTS mykey
    DEL nonexistentkey // A response of (integer) 0 indicates that no keys were deleted (because the key didn't exist)
    DEL user:1001 nonexistentkey >> integer 1
    SET mykey myvalue
    SET user:1001 John
    DEL mykey user:1001 >> (integer) 2

* redis-cli
    SET sessionkey sessionvalue EX 15
    GET sessionkey
    TTL sessionkey

    SET tempkey tempvalue
    GET tempkey
    TTL tempkey
    EXPIRE tempkey 10
    TTL tempkey >> Redis will respond with the number of seconds remaining until the key expires
    

# **HomeLab Builds**

### App Installs

sudo pacman -Syy << That forces pacman to redownload all repository databases
sudo pacman -Syu
sudo cachyos-rate-mirrors

sudo pacman -S dbeaver
    dbeaver
    

### Nagios

localhost.cfg  → what Nagios monitors
commands.cfg   → how Nagios performs the checks
libexec/       → actual monitoring plugins
nagios.cfg     → main Nagios configuration
Apache         → Nagios web interface

#### Nagios .cfg Configs

##### command.cfg 
vim /usr/local/nagios/etc/objects/commands.cfg

define command {
    command_name    check_icmp
    command_line    $USER1$/check_icmp -H $HOSTADDRESS$ -w $ARG1$ -c $ARG2$
}

define command {
    command_name    check-host-alive
    command_line    $USER1$/check_icmp -H $HOSTADDRESS$ -w 3000.0,80% -c 5000.0,100%
}

#### Nagios Host Configs

##### Localhost
/usr/local/nagios/etc/objects/localhost.cfg
vim /usr/local/nagios/etc/objects/localhost.cfg

* troubleshooting: CRITICAL - PING You need more args!!!
grep -A5 'command_name.*check_ping' /usr/local/nagios/etc/objects/commands.cfg
grep -A10 'service_description.*PING' /usr/local/nagios/etc/objects/localhost.cfg
service_description     PING
check_command           check_ping!100.0,20%!500.0,60%
    This passes two arguments:
        $ARG1$ = 100.0,20% << warning: means WARNING if either average round-trip time (RTA) exceeds 100 ms, or packet loss exceeds 20%
        $ARG2$ = 500.0,60% << critcal: means CRITICAL if either average round-trip time exceeds 500 ms, or packet loss exceeds 60%
So now we need to inspect how check_ping is defined in commands.cfg
    grep -A8 'command_name.*check_ping' /usr/local/nagios/etc/objects/commands.cfg
    /usr/local/nagios/libexec/check_ping -H 127.0.0.1 -w 100.0,20% -c 500.0,60% -p 5
        -H 127.0.0.1 \ << host
        -w 100.0,20% \ << warning
        -c 500.0,60% \ << critical
        -p 5 << pass
    ls -l /usr/bin/ping
    getcap /usr/bin/ping
    sudo ping -c 3 127.0.0.1
    sudo setcap cap_net_raw+ep /usr/bin/ping
    sudo chmod u+s /bin/ping
    getcap /usr/bin/ping << /usr/bin/ping cap_net_raw=ep
    /usr/local/nagios/libexec/check_ping --help
    strings /usr/local/nagios/libexec/check_ping | grep -E '/(bin|sbin)/ping'
    grep -i 'PING' config.log | grep -E 'PATH|ping command|ping -'
    grep -E '^(PING|PING6)' plugins/Makefile | head -20
    grep -R 'define PING' plugins/ 2>/dev/null | head

* troubleshooting: HTTP WARNING: HTTP/1.1 403 Forbidden
grep -A10 'service_description.*HTTP' /usr/local/nagios/etc/objects/localhost.cfg


* troubleshooting: WARNING - load average: 5.37, 3.78, 2.77
grep -A10 'service_description.*Current Load' /usr/local/nagios/etc/objects/localhost.cfg
service_description     Current Load
check_command           check_local_load!5.0,4.0,3.0!10.0,6.0,4.0
    WARNING:  5.0, 4.0, 3.0
    CRITICAL: 10.0, 6.0, 4.0
        Nagios reported:
            WARNING - load average: 5.37, 3.78, 2.77
        Your 1-minute load was 5.37, and your warning threshold is 5.0

Find: /usr/local/nagios/etc/objects/localhost.cfg
define service {
    use                     local-service
    host_name               localhost
    service_description     PING
    check_command           check_ping!100.0,20%!500.0,60%
}

Change only the check_command line:
check_command           check_icmp!100.0,20%!500.0,60%

/usr/local/nagios/bin/nagios -v /usr/local/nagios/etc/nagios.cfg

#### Nagios Installation and Setup

##### Intialization setup
Nagios Core = schedules checks, stores results, handles alerts, displays the UI.
Nagios Plugins = programs that actually perform checks like check_ping, check_http, check_load, check_users, check_disk, etc

cat /etc/os-release
hostname -I
sudo hostnamectl set-hostname nagios-server
hostnamectl 

##### Installing neccessary componets
* sudo dnf update -y
* sudo dnf install -y tar gcc glibc glibc-common wget unzip httpd php gd gd-devel perl openssl-devel
    We're installing a few different categories here: 
        gcc/glibc: for compiling Nagios
        httpd: for the Apache web interface
        php: for web functionality
        gd: for graphics
        openssl-devel: plus perl for supporting components

##### Step 3 — Download && Extract Nagios Core
cd /tmp
wget --output-document="nagioscore.tar.gz" $(wget -q -O - https://api.github.com/repos/NagiosEnterprises/nagioscore/releases/latest  | grep '"browser_download_url":' | grep -o 'https://[^"]*')
ls -lh nagioscore.tar.gz
file nagioscore.tar.gz
* tar xzf nagioscore.tar.gz
    x = extract
    z = decompress gzip
    f = use the following file

##### Step 4 — Configure the build
ls -ld nagios-*
cd nagios-*/
* ./configure --with-httpd-conf=/etc/httpd/conf.d
    --with-httpd-conf=/etc/httpd/conf.d: 
        tells Nagios: When we eventually install the Nagios Apache configuration, this is where CentOS expects Apache's additional configuration files

##### Step 5 — Compile Nagios
make all

##### Step 6 — Create the Nagios service account
sudo make install-groups-users
id nagios
sudo usermod -aG nagios apache >> The Apache user on CentOS is apache. Eventually Apache needs permission to interact with parts of Nagios, so add it to the nagios group

##### Step 7 — Install Nagios Core
sudo make install 
ls -lh /usr/local/nagios/bin/nagios << this verifies the main binary exists
/usr/local/nagios/bin/nagios --version

##### Step 8 -- Next install the systemd service definition
sudo make install-daemoninit
systemctl status nagios

##### Troubleshooting permission denied error /usr/local/nagios/bin/nagios --version
ls -l /usr/local/nagios/bin/nagios
namei -l /usr/local/nagios/bin/nagios
getenforce << selinux verification
sudo usermod -aG nagios sysadmin
groups sysadmin << You should see nagios added
exit
ssh sysadmin@10.0.0.90

##### Step 9 — Install the Nagios command mode
cd /tmp/nagios-4.5.14
sudo make install-commandmode

##### Step 10 — Install the sample configuration
sudo make install-config
ls -lh /usr/local/nagios/etc/
    You should see files such as:
        cgi.cfg
        nagios.cfg
        resource.cfg
        objects/

##### Step 11 — Install the Apache configuration
sudo make install-webconf
ls -l /etc/httpd/conf.d/ << You should see something Nagios-related, likely: nagios.conf

##### Step 12 — Create the web UI account
sudo htpasswd -c /usr/local/nagios/etc/htpasswd.users nagiosadmin
    The -c means create a new password file. That's important because you generally only use -c for the first account  If you later add another user, you'd use:
        sudo htpasswd /usr/local/nagios/etc/htpasswd.users sysadmin
cat /usr/local/nagios/etc/htpasswd.users 

##### Step 12 — Validate Apache
sudo apachectl configtest << Syntax OK is what you want to see

##### Step 13 — Validate Nagios
/usr/local/nagios/bin/nagios -v /usr/local/nagios/etc/nagios.cfg
    You'll get quite a bit of output. Near the bottom we're looking for:
        Total Warnings: 0
        Total Errors:   0

        Things look okay - No serious problems were detected during the pre-flight check
    
##### Step 14 — Start Apache and Nagios
sudo systemctl enable --now httpd
sudo systemctl enable --now nagios
systemctl status httpd --no-pager
systemctl status nagios --no-pager

##### Step 15 — Check the CentOS firewall
sudo firewall-cmd --state
sudo firewall-cmd --permanent --add-service=http
sudo firewall-cmd --reload
sudo firewall-cmd --list-services

##### Step 16 — Open Nagios
http://10.0.0.90/nagios/

#### Nagios Plugin Setups

##### Installing Nagios Pluins prereqs
ls -lh /usr/local/nagios/libexec/
sudo dnf install -y autoconf automake gcc glibc libtool make openssl-devel wget gettext net-snmp net-snmp-utils perl-Net-SNMP bind-utils iputils
cd /tmp
wget --output-document="nagios-plugins.tar.gz" $(wget -q -O - https://api.github.com/repos/nagios-plugins/nagios-plugins/releases/latest | grep '"browser_download_url":' | grep -o 'https://[^"]*' | head -1)
ls -lh nagios-plugins.tar.gz
file nagios-plugins.tar.gz << gzip compressed data
ls -d nagios-plugins-*/

##### Configure the plugins
* ./configure --with-nagios-user=nagios --with-nagios-group=nagios
    This checks your system for the libraries/tools each plugin needs and prepares the Makefiles

##### Complie the plugins
If configuration finishes successfully, compile them:
make 
sudo make install
ls -lh /usr/local/nagios/libexec/ << You should now see a whole bunch of plugins
ls -l \
/usr/local/nagios/libexec/check_load \
/usr/local/nagios/libexec/check_users \
/usr/local/nagios/libexec/check_http \
/usr/local/nagios/libexec/check_ping

##### Plugin Test
/usr/local/nagios/libexec/check_load -w 15,10,5 -c 30,25,20
    -w means WARNING
    -c means CRITICAL
    Those are the system's 1-minute, 5-minute, and 15-minute load averages

##### Now let Nagios use the plugins
sudo systemctl restart nagios
systemctl status nagios --no-pager

# **k8s**

### Definitions

#### ConfigMap
stores non-sensitive configuration data for applications
literally the configs for the pod so that way you can customize and update it at will


### Commands

#### cluster management
kubectl cluster-info
kubectl cluster-info dump > clusterinfo
kubectl api-resources
kubectl api-resources | grep -i pod
kubectl config set-context --current --namespace=ckad-prep-contexts
kubectl wait --namespace kube-system --for=condition=Ready pod -l k8s-app=calico-node --timeout=120sss

#### kubectl config
kubectl config current-context
kubectl config set-context --current --namespace=default
kubectl config view --minify --output 'jsonpath={..namespace}'; echo
kubectl config view | grep -i namespace

#### kubectl get
kubectl get nodes --request-timeout=5s
kubectl get no -o wide
kubectl get namespaces
kubectl get configmap
kubectl get configmap app-settings --namespace default --ignore-not-found
kubectl get jobs
kubectl get job heartbeat-test
kubectl get pods -l job-name=heartbeat-test
kubectl get pods --show-labels
kubectl get pods -l app=web-demo --show-labels
kubectl get pods -l app=web-demo -o 'custom-columns=NAME:.metadata.name,OWNER:.metadata.ownerReferences[0].kind,PHASE:.status.phase'
kubectl get pods -n kube-system
kubectl get pods -n grafana -w
kubectl get deployment web-demo
kubectl get deployment web-demo -o wide
kubectl get deploy web-demo -o yaml > lab_deploymnet.yaml
kubectl get deployment web-demo -o 'jsonpath={.spec.selector.matchLabels.app}{" "}{.spec.template.metadata.labels.app}{"\n"}'
kubectl get replicasets -n grafana
kubec

#### kubectl create and delete
* the key difference between a single Pod and a Deployment -- If you delete standalone Pods, they are gone. If you delete Pods managed by a Deployment, the Deployment notices that the actual state no longer matches the desired state and creates replacements
kubectl create namespace ckad-prep-contexts
kubectl create configmap app-settings --from-literal=mode=prep
kubectl create job --from=cronjob/cluster-heartbeat heartbeat-test
kubectl delete pods -l app=web-demo --wait=false
kubectl delete pods -l app=web-demo --wait=false && kubectl get pods -w

#### kubectl logs
kubectl logs job/heartbeat-test
kubectl logs nginx -n dev

#### kubectl describe
kubectl describe job heartbeat-test
kubectl describe rs web-demo-8669cb4554 -n grafana
kubectl describe rs web-demo-8669cb4554 -n grafana | grep -i labels


#### kubeclt create deployment
kubectl create deployment web-demo --image=registry.k8s.io/pause:3.10.1 --replicas=2
kubectl rollout status deployment/web-demo --timeout=90s
kubectl get deployment web-demo


### Manifests and Deployments

### CKAD Training

#### Explore kubectl,Contexts, and Namespaces
mkdir -p /home/labex/project
kubectl config current-context
kubectl config set-context --current --namespace=default
kubectl get nodes --request-timeout=5s
kubectl cluster-info
kubectl get namespaces
kubectl config view --minify --output 'jsonpath={..namespace}'; echo
kubectl get pods --namespace kube-system
kubectl create namespace ckad-prep-contexts
kubectl config set-context --current --namespace=ckad-prep-contexts
kubectl config view --minify --output 'jsonpath={..namespace}'; echo
kubectl get pods --namespace kube-system
kubectl create configmap app-settings --from-literal=mode=prep
kubectl get configmap app-settings --namespace ckad-prep-contexts

#### Inspect k8s API Resources

### **Manifest Files**

#### heartbeat-cronjob.yaml

apiVersion: batch/v1
kind: CronJob
metadata:
  name: cluster-heartbeat
spec:
  schedule: "*/5 * * * *" #*/5 meant "every 5 minutes."
  schedule: "0 * * * *"  #0 means "exactly at minute 0" (the start of the hour).
  schedule: "0 8 * * *"


  jobTemplate:
    spec:
      template:
        spec:
          containers:
            - name: heartbeat
              image: busybox:1.37
              command:
                - /bin/sh
                - -c
                - 'echo "Cluster heartbeat: $(date)"'

          restartPolicy: OnFailure

kubectl apply -f heartbeat-cronjob.yaml
kubectl get cronjobs
kubectl create job --from=cronjob/cluster-heartbeat heartbeat-test
kubectl get jobs
kubectl get pods
kubectl delete job heartbeat-test

# **LInux**

### Cronjobs

#### syntax
0 (Minute) → Exactly at minute 0
8 (Hour) → Exactly at hour 8 (8:00 AM)
* (Day of Month) → Every day
* (Month) → Every month
* (Day of Week) → Every day of the week

### Vim Motions

/search >> after it is highligted
:noh

# **Networking**

### Switch Commands

#### Initial Setup
en
sh ip int br
sh int status
sh vlan br
sh run
wr
sh start

conf t
    hostname LanSW1
    no ip domain-lookup
    banner motd @Warning! Unauthorized Access is Prohibited@
    service password encryption
    enable secret ccna
    username admin password admin
    username sysadmin privilege 15 password admin123
    ip domain-name MasonIT
    crypto key generate rsa 1024
    ip ssh version 2


    line console 0
        password Passw0rd
        login
        logging synchronous
        do wr
        do sh run

    line vty 0 15
        login local
        login
        exec timeout 0 0
        logging synchronous
        do wr
        do sh run

#### Interface Setup
en 
conf t
    int g0/1
        switchport mode access
        switchport access vlan 10
        no shut
        shut

conf t
    int g0/3
        no shut
        end

#### Vlans
en
sh vlan br

conf t
    vlan 10
        name lab1
    no vlan 10

int vlan1
    ip address 10.0.0.254 255.255.255.0
    no shut

# **Proxmox**

### Commands

pct list
pct set <CTID> --hostname nagios-server
pct set 105 --hostname nagios-server